# ECS111 Final Project — Full Run on Colab (T4 GPU), self-contained

**No git clone needed.** This notebook carries all the project code inside it.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. (Optional) set `SHARD` in the next code cell to split the work with teammates.
3. Runtime -> **Run all**. If Colab disconnects, just Run all again — the runner
   resumes and skips finished work.

`SHARD` options (one teammate each for a ~30-min run, or leave `all` for one
~2-hour run): `all` · `baseline` · `cot_plain` · `cot_structured` ·
`finetune_answers` · `finetune_traces`.

When it finishes, download `ecs111_results.zip` and send it back, or paste the
contents of `results/summary_table.md`.


In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA available:", True, "|", torch.cuda.get_device_name(0))
else:
    print("!!! No GPU detected.")
    print("!!! Runtime -> Change runtime type -> T4 GPU, then Run all again.")


In [ ]:
SHARD = "all"  # all | baseline | cot_plain | cot_structured | finetune_answers | finetune_traces
import os
for d in ("src", "scripts", "results", "checkpoints"):
    os.makedirs(d, exist_ok=True)
print("SHARD =", SHARD)


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""Single source of truth for models, datasets, hyperparameters, and paths.

Pure-Python and dependency-light on purpose: importing this module must NOT
require torch/transformers, so the pure-logic modules (metrics, data
serialization, traces) can import it without the heavy ML stack. `get_device`
imports torch lazily, only when actually called.
"""

from __future__ import annotations

import os
from pathlib import Path

# --------------------------------------------------------------------------- #
# Reproducibility
# --------------------------------------------------------------------------- #
SEEDS = [13, 42]  # every condition runs twice; report mean +/- std

# --------------------------------------------------------------------------- #
# Models (HuggingFace ids)
# --------------------------------------------------------------------------- #
FLAN_SMALL = "google/flan-t5-small"  # 80M  -- SMOKE / fast local checks only
FLAN_BASE = "google/flan-t5-base"    # 250M -- fine-tuned + prompted
FLAN_LARGE = "google/flan-t5-large"  # 780M -- prompting only (large OOMs when fine-tuned on T4)

# Conditions that fine-tune use base only; prompting conditions use both.
PROMPT_MODELS = [FLAN_BASE, FLAN_LARGE]
FINETUNE_MODEL = FLAN_BASE

# --------------------------------------------------------------------------- #
# Datasets (HuggingFace ids) -- verified by real download (see scripts/smoke_local.py)
#
# NOTE: `datasets` >= 4.0 removed support for script-based datasets, so the
# canonical `wikitablequestions` and `tab_fact` ids no longer load. We use
# parquet-native mirrors with identical content:
#   * WTQ     -> lighteval/wikitablequestions  (inline header+rows tables;
#                single 18,486-example pool -> we make a fixed SEEDED disjoint
#                train/eval partition, so train and eval never overlap).
#   * TabFact -> target-benchmark/{tabfact-queries, tabfact-corpus}
#                (queries carry statement + True/False; corpus carries the
#                 table as a list-of-lists, joined on table_id).
# --------------------------------------------------------------------------- #
WTQ_ID = "lighteval/wikitablequestions"          # Pasupat & Liang 2015, primary
TABFACT_QUERIES_ID = "target-benchmark/tabfact-queries"  # Chen et al. 2020, OOD test
TABFACT_CORPUS_ID = "target-benchmark/tabfact-corpus"

# --------------------------------------------------------------------------- #
# Evaluation sizing (seeded caps keep every condition < 2 hr on a Colab T4)
# --------------------------------------------------------------------------- #
EVAL_N = 1000        # baseline / fine-tune / TabFact eval slice
EVAL_N_COT = 500     # CoT prompting is slower (long prompts) -> smaller slice
EVAL_SEED = 13       # seed used to sample the eval slice (held fixed across conditions)

# --------------------------------------------------------------------------- #
# Fine-tuning hyperparameters (proposal section 5)
# --------------------------------------------------------------------------- #
LEARNING_RATE = 3e-4
TRAIN_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4          # effective batch size = 8 * 4 = 32
EPOCHS = 3
TRAIN_N = 8000               # cap on fine-tune training examples (keeps a run < 2 hr on a T4)
N_SHOTS = 6                  # CoT exemplars prepended per prompt (proposal: 6-8)
MAX_SOURCE_LEN = 512          # baseline + fine-tune source (table + question)
MAX_SOURCE_LEN_PROMPT = 1024  # CoT prompts are long (6 exemplars); give the encoder more room
MAX_TARGET_LEN = 128          # answer, or reasoning chain + answer
GEN_MAX_NEW_TOKENS = 128

# Decoding: greedy, temperature 0 (deterministic) for all reported runs.
GREEDY = True

# --------------------------------------------------------------------------- #
# SMOKE mode -- tiny real end-to-end run to prove the pipeline executes.
# Used both as the first Colab sanity cell and for local MPS verification.
# --------------------------------------------------------------------------- #
SMOKE_MODEL = FLAN_SMALL
SMOKE_TRAIN_N = 16
SMOKE_EVAL_N = 8
SMOKE_EPOCHS = 1
SMOKE_MAX_STEPS = 2

# --------------------------------------------------------------------------- #
# Error-type labels (proposal section 6)
# --------------------------------------------------------------------------- #
ERROR_TYPES = ["lookup", "aggregation", "multi_hop", "correct"]

# --------------------------------------------------------------------------- #
# Paths
# --------------------------------------------------------------------------- #
REPO_ROOT = Path(__file__).resolve().parent.parent
RESULTS_DIR = REPO_ROOT / "results"
CHECKPOINTS_DIR = REPO_ROOT / "checkpoints"


def get_device() -> str:
    """Return the best available torch device: cuda (Colab) > mps (Apple) > cpu.

    Imports torch lazily so this module stays importable without the ML stack.
    """
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        # Let unsupported MPS ops fall back to CPU instead of crashing.
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return "mps"
    return "cpu"


def eval_n_for(condition: str, smoke: bool = False) -> int:
    """Eval slice size for a condition. CoT gets the smaller cap."""
    if smoke:
        return SMOKE_EVAL_N
    return EVAL_N_COT if "cot" in condition.lower() else EVAL_N


In [ ]:
%%writefile src/data.py
"""Dataset loading + table serialization.

Unified example schema used everywhere downstream:

    {
        "id":       str,
        "task":     "wtq" | "tabfact",
        "question": str,                       # WTQ question  OR  TabFact statement
        "table":    {"header": [str, ...],
                     "rows":   [[str, ...], ...]},
        "answer":   str,                       # WTQ: gold answer text
                                               # TabFact: "true" | "false"
        "answers":  [str, ...],                # WTQ raw gold list (>=1); TabFact: [answer]
    }

`serialize_table` and the TabFact text parser are pure Python (no `datasets`
needed) so they are unit-testable offline. The HF loaders import `datasets`
lazily inside the function, so importing this module never requires the ML stack.
"""

from __future__ import annotations

import ast
import random
from typing import Any

from . import config


# --------------------------------------------------------------------------- #
# Table serialization  (proposal: "headers first, then each row in sequence")
# --------------------------------------------------------------------------- #
def serialize_table(table: dict[str, Any]) -> str:
    """Flatten a {header, rows} table to a compact, model-friendly string.

    Format: header line then one line per row, cells joined by " | ".
    """
    header = [str(h) for h in table["header"]]
    lines = [" | ".join(header)]
    for row in table["rows"]:
        lines.append(" | ".join(str(c) for c in row))
    return "\n".join(lines)


# --------------------------------------------------------------------------- #
# TabFact corpus table parser  (pure)
# --------------------------------------------------------------------------- #
def parse_corpus_table(table: Any) -> dict[str, list]:
    """Convert a TabFact-corpus table (list-of-lists, header first) to {header, rows}.

    Accepts either a real list or its string repr (defensive).
    """
    if isinstance(table, str):
        table = ast.literal_eval(table)
    if not table:
        return {"header": [], "rows": []}
    header = [str(c) for c in table[0]]
    rows = [[str(c) for c in r] for r in table[1:]]
    return {"header": header, "rows": rows}


# --------------------------------------------------------------------------- #
# Seeded sampling  (pure)
# --------------------------------------------------------------------------- #
def sample_indices(n_total: int, n_take: int, seed: int) -> list[int]:
    """Deterministic sample of up to n_take indices from range(n_total)."""
    if n_take >= n_total:
        return list(range(n_total))
    rng = random.Random(seed)
    return sorted(rng.sample(range(n_total), n_take))


def disjoint_train_eval_indices(
    total: int, eval_n: int, eval_seed: int, train_n: int, train_seed: int
) -> tuple[list[int], list[int]]:
    """Fixed eval slice first, then a train slice drawn from the remainder.

    Guarantees train and eval indices never overlap, regardless of seeds. Used
    to carve a clean train/eval partition out of the single WTQ pool (the
    parquet mirror ships one split; see config note).
    """
    eval_idxs = sample_indices(total, eval_n, eval_seed)
    eval_set = set(eval_idxs)
    pool = [i for i in range(total) if i not in eval_set]
    if train_n >= len(pool):
        train_idxs = pool
    else:
        rng = random.Random(train_seed)
        train_idxs = sorted(rng.sample(pool, train_n))
    return train_idxs, eval_idxs


# --------------------------------------------------------------------------- #
# Normalizers (raw HF row -> unified schema)
# --------------------------------------------------------------------------- #
def _normalize_wtq(row: dict[str, Any], idx: int) -> dict[str, Any]:
    answers = [str(a) for a in row.get("answers", [])]
    return {
        "id": str(row.get("id", f"wtq-{idx}")),
        "task": "wtq",
        "question": row["question"],
        "table": {
            "header": [str(h) for h in row["table"]["header"]],
            "rows": [[str(c) for c in r] for r in row["table"]["rows"]],
        },
        "answer": ", ".join(answers),
        "answers": answers,
    }


def _normalize_tabfact(query_row: dict[str, Any], table: dict[str, list], idx: int) -> dict[str, Any]:
    answer = "true" if str(query_row["answer"]).strip().lower() == "true" else "false"
    return {
        "id": str(query_row.get("query_id", f"tabfact-{idx}")),
        "task": "tabfact",
        "question": query_row["query"],   # the statement to verify
        "table": table,
        "answer": answer,
        "answers": [answer],
    }


# --------------------------------------------------------------------------- #
# HF loaders  (import `datasets` lazily)
# --------------------------------------------------------------------------- #
def _wtq_pool():
    """The single WTQ pool from the parquet mirror (inline tables)."""
    from datasets import load_dataset

    return load_dataset(config.WTQ_ID, split="test")


def load_wtq_eval(n: int | None = config.EVAL_N, seed: int = config.EVAL_SEED) -> list[dict]:
    """Seeded WTQ evaluation slice (None = full pool)."""
    ds = _wtq_pool()
    idxs = range(len(ds)) if n is None else sample_indices(len(ds), n, seed)
    return [_normalize_wtq(ds[int(i)], int(i)) for i in idxs]


def load_wtq_train(
    n: int,
    seed: int,
    eval_n: int = config.EVAL_N,
    eval_seed: int = config.EVAL_SEED,
) -> list[dict]:
    """Seeded WTQ training slice, guaranteed disjoint from the eval slice."""
    ds = _wtq_pool()
    train_idxs, _ = disjoint_train_eval_indices(len(ds), eval_n, eval_seed, n, seed)
    return [_normalize_wtq(ds[int(i)], int(i)) for i in train_idxs]


def load_tabfact(split: str = "test", n: int | None = config.EVAL_N, seed: int = config.EVAL_SEED) -> list[dict]:
    """Load TabFact (OOD test), joining queries to corpus tables on table_id."""
    from datasets import load_dataset

    queries = load_dataset(config.TABFACT_QUERIES_ID, split=split)
    corpus = load_dataset(config.TABFACT_CORPUS_ID, split=split)
    table_by_id = {row["table_id"]: parse_corpus_table(row["table"]) for row in corpus}

    idxs = range(len(queries)) if n is None else sample_indices(len(queries), n, seed)
    out = []
    for i in idxs:
        q = queries[int(i)]
        table = table_by_id.get(q["table_id"])
        if table is None:
            continue
        out.append(_normalize_tabfact(q, table, int(i)))
    return out


In [ ]:
%%writefile src/prompts.py
"""Prompt builders + answer extraction.

One baseline format is shared by (a) zero-shot baseline inference, (b) the
fine-tuning source text (Conditions B & C), and (c) the generalization pass on
TabFact (statement fed as the question). The CoT builder prepends hand-written
exemplars. Keeping a single source format means a model fine-tuned on WTQ sees
the same surface form when evaluated on TabFact.
"""

from __future__ import annotations

from . import config
from .cot_exemplars import format_exemplar, get_exemplars
from .data import serialize_table

BASELINE_INSTRUCTION = "Answer the question based on the table."
ANSWER_TAG = "Answer:"


def build_baseline_prompt(example: dict) -> str:
    """Zero-shot / fine-tune source: instruction + question + serialized table."""
    return "\n".join(
        [
            BASELINE_INSTRUCTION,
            f"Question: {example['question']}",
            "Table:",
            serialize_table(example["table"]),
            ANSWER_TAG,
        ]
    )


TABFACT_INSTRUCTION = "Read the table and decide whether the statement is true or false."


def build_tabfact_prompt(example: dict) -> str:
    """Zero-shot TabFact prompt: instruction + statement + table, asking for true/false.

    TabFact is true/false verification, not open QA. Feeding it through the WTQ
    'answer the question' prompt never tells the model to output true or false,
    so the generation is unmappable and the score collapses. This prompt asks for
    the label directly; no TabFact training happens, so it stays a fair OOD test.
    """
    return "\n".join(
        [
            TABFACT_INSTRUCTION,
            f"Statement: {example['question']}",
            "Table:",
            serialize_table(example["table"]),
            "Answer (true or false):",
        ]
    )


def build_cot_prompt(example: dict, style: str = "plain", n_shots: int = 6) -> str:
    """Few-shot CoT: n exemplars (with reasoning) then the query ending in 'Reasoning:'.

    style in {"plain", "structured"} selects the chain format (proposal Condition A).
    """
    shots = get_exemplars(style)[:n_shots]
    blocks = [format_exemplar(ex, style) for ex in shots]
    # Put the question right before "Reasoning:" so it survives left-truncation
    # of long prompts (the table can be big; the question must not be cut).
    query = "\n".join(
        [
            "Table:",
            serialize_table(example["table"]),
            f"Question: {example['question']}",
            "Reasoning:",
        ]
    )
    return "\n\n".join(blocks + [query])


def extract_answer(generated_text: str) -> str:
    """Pull the final answer out of a model generation.

    CoT outputs end with 'Answer: X'; take the text after the last tag. Plain
    generations (no tag) are returned stripped as-is.
    """
    if ANSWER_TAG in generated_text:
        return generated_text.rsplit(ANSWER_TAG, 1)[-1].strip()
    return generated_text.strip()


def build_train_source(example: dict) -> str:
    """Fine-tuning source text (same as the baseline prompt)."""
    return build_baseline_prompt(example)


def build_train_target(example: dict, trace: str | None = None) -> str:
    """Fine-tuning target.

    Condition B (answers-only): target is the gold answer.
    Condition C (with trace):   target is the reasoning chain (which already
                                ends in 'Answer: <gold>').
    """
    if trace is not None:
        return trace
    return example["answer"]


In [ ]:
%%writefile src/cot_exemplars.py
"""Hand-crafted Chain-of-Thought exemplars for table question answering.

Each exemplar is a dict with keys:
    question : str
    table    : {"header": [str, ...], "rows": [[str, ...], ...]}
    reasoning: str
    answer   : str

Two parallel lists are provided:
    EXEMPLARS_PLAIN      -- natural-language paragraph reasoning
    EXEMPLARS_STRUCTURED -- numbered step-by-step reasoning

Every answer has been independently verified against its table.
"""

from __future__ import annotations

from .data import serialize_table

# ---------------------------------------------------------------------------
# EXEMPLARS_PLAIN
# ---------------------------------------------------------------------------
EXEMPLARS_PLAIN: list[dict] = [
    # 1. Simple cell lookup
    {
        "question": "What is Alice's score?",
        "table": {
            "header": ["Name", "Score"],
            "rows": [
                ["Alice", "92"],
                ["Bob", "78"],
                ["Carol", "85"],
            ],
        },
        "reasoning": (
            "I need to find the row where the Name column equals 'Alice'. "
            "Looking through the rows, the first row has Name = 'Alice' and Score = '92'. "
            "Therefore Alice's score is 92."
        ),
        "answer": "92",
    },
    # 2. Row filtering by condition
    {
        "question": "Which city has a population greater than 2 million?",
        "table": {
            "header": ["City", "Population (millions)"],
            "rows": [
                ["Springfield", "0.5"],
                ["Shelbyville", "2.3"],
                ["Capital City", "1.1"],
            ],
        },
        "reasoning": (
            "I need to filter rows where the population is greater than 2 million. "
            "Springfield has 0.5 million, which is not greater than 2. "
            "Shelbyville has 2.3 million, which is greater than 2, so it qualifies. "
            "Capital City has 1.1 million, which is not greater than 2. "
            "Only Shelbyville satisfies the condition."
        ),
        "answer": "Shelbyville",
    },
    # 3. Counting
    {
        "question": "How many products cost more than $20?",
        "table": {
            "header": ["Product", "Price"],
            "rows": [
                ["Widget", "15"],
                ["Gadget", "25"],
                ["Doohickey", "30"],
                ["Thingamajig", "10"],
            ],
        },
        "reasoning": (
            "I need to count how many rows have Price greater than 20. "
            "Widget costs 15, which is not more than 20. "
            "Gadget costs 25, which is more than 20. "
            "Doohickey costs 30, which is more than 20. "
            "Thingamajig costs 10, which is not more than 20. "
            "Two products (Gadget and Doohickey) cost more than $20."
        ),
        "answer": "2",
    },
    # 4. Aggregation: sum
    {
        "question": "What is the total revenue across all months?",
        "table": {
            "header": ["Month", "Revenue"],
            "rows": [
                ["January", "1200"],
                ["February", "1500"],
                ["March", "1100"],
            ],
        },
        "reasoning": (
            "I need to sum the Revenue column for all rows. "
            "January contributes 1200, February contributes 1500, and March contributes 1100. "
            "Adding these together: 1200 + 1500 = 2700, then 2700 + 1100 = 3800. "
            "The total revenue is 3800."
        ),
        "answer": "3800",
    },
    # 5. Aggregation: max
    {
        "question": "What is the highest temperature recorded?",
        "table": {
            "header": ["City", "High Temp (F)"],
            "rows": [
                ["Phoenix", "118"],
                ["Las Vegas", "115"],
                ["Death Valley", "130"],
                ["Tucson", "112"],
            ],
        },
        "reasoning": (
            "I need to find the maximum value in the High Temp (F) column. "
            "The temperatures are 118 for Phoenix, 115 for Las Vegas, 130 for Death Valley, "
            "and 112 for Tucson. "
            "The highest among these is 130, recorded in Death Valley."
        ),
        "answer": "130",
    },
    # 6. Comparison
    {
        "question": "Does team Lions have more wins than team Tigers?",
        "table": {
            "header": ["Team", "Wins", "Losses"],
            "rows": [
                ["Lions", "10", "4"],
                ["Tigers", "7", "7"],
                ["Bears", "10", "4"],
            ],
        },
        "reasoning": (
            "I need to compare the Wins values for Lions and Tigers. "
            "Looking up Lions, they have 10 wins. "
            "Looking up Tigers, they have 7 wins. "
            "Since 10 is greater than 7, Lions have more wins than Tigers."
        ),
        "answer": "yes",
    },
    # 7. Sorting / superlative: min
    {
        "question": "Which country has the smallest area?",
        "table": {
            "header": ["Country", "Area (sq km)"],
            "rows": [
                ["Brazil", "8515767"],
                ["Canada", "9984670"],
                ["Australia", "7692024"],
                ["India", "3287263"],
            ],
        },
        "reasoning": (
            "I need to find the country with the minimum value in the Area (sq km) column. "
            "The areas are: Brazil 8,515,767; Canada 9,984,670; Australia 7,692,024; India 3,287,263. "
            "Comparing all four, India's area of 3,287,263 is the smallest. "
            "Therefore India has the smallest area."
        ),
        "answer": "India",
    },
    # 8. Multi-hop: filter to cheapest, then look up its rating
    {
        "question": "What is the rating of the cheapest product?",
        "table": {
            "header": ["Product", "Price", "Rating"],
            "rows": [
                ["Alpha", "50", "4.2"],
                ["Beta", "30", "3.8"],
                ["Gamma", "45", "4.5"],
            ],
        },
        "reasoning": (
            "This is a two-step question. "
            "First, I find the cheapest product by looking at the Price column: "
            "Alpha costs 50, Beta costs 30, and Gamma costs 45. "
            "The minimum price is 30, so the cheapest product is Beta. "
            "Second, I look up Beta's rating, which is 3.8. "
            "Therefore the rating of the cheapest product is 3.8."
        ),
        "answer": "3.8",
    },
]

# ---------------------------------------------------------------------------
# EXEMPLARS_STRUCTURED
# ---------------------------------------------------------------------------
EXEMPLARS_STRUCTURED: list[dict] = [
    # 1. Simple cell lookup
    {
        "question": "What is Alice's score?",
        "table": {
            "header": ["Name", "Score"],
            "rows": [
                ["Alice", "92"],
                ["Bob", "78"],
                ["Carol", "85"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the target column. The question asks for a Score value.\n"
            "Step 2: Filter rows by Name = 'Alice'. The matching row is ['Alice', '92'].\n"
            "Step 3: Read the Score value from that row: 92.\n"
            "Conclusion: Alice's score is 92."
        ),
        "answer": "92",
    },
    # 2. Row filtering by condition
    {
        "question": "Which city has a population greater than 2 million?",
        "table": {
            "header": ["City", "Population (millions)"],
            "rows": [
                ["Springfield", "0.5"],
                ["Shelbyville", "2.3"],
                ["Capital City", "1.1"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the filtering condition: Population (millions) > 2.\n"
            "Step 2: Check each row.\n"
            "  - Springfield: 0.5 > 2? No.\n"
            "  - Shelbyville: 2.3 > 2? Yes.\n"
            "  - Capital City: 1.1 > 2? No.\n"
            "Step 3: Collect cities that pass: Shelbyville.\n"
            "Conclusion: Shelbyville is the only city with a population greater than 2 million."
        ),
        "answer": "Shelbyville",
    },
    # 3. Counting
    {
        "question": "How many products cost more than $20?",
        "table": {
            "header": ["Product", "Price"],
            "rows": [
                ["Widget", "15"],
                ["Gadget", "25"],
                ["Doohickey", "30"],
                ["Thingamajig", "10"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the filtering condition: Price > 20.\n"
            "Step 2: Check each row.\n"
            "  - Widget: 15 > 20? No.\n"
            "  - Gadget: 25 > 20? Yes. (count = 1)\n"
            "  - Doohickey: 30 > 20? Yes. (count = 2)\n"
            "  - Thingamajig: 10 > 20? No.\n"
            "Step 3: Total count of qualifying rows = 2.\n"
            "Conclusion: 2 products cost more than $20."
        ),
        "answer": "2",
    },
    # 4. Aggregation: sum
    {
        "question": "What is the total revenue across all months?",
        "table": {
            "header": ["Month", "Revenue"],
            "rows": [
                ["January", "1200"],
                ["February", "1500"],
                ["March", "1100"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the column to sum: Revenue.\n"
            "Step 2: Collect all Revenue values: 1200, 1500, 1100.\n"
            "Step 3: Add them: 1200 + 1500 = 2700; 2700 + 1100 = 3800.\n"
            "Conclusion: The total revenue is 3800."
        ),
        "answer": "3800",
    },
    # 5. Aggregation: max
    {
        "question": "What is the highest temperature recorded?",
        "table": {
            "header": ["City", "High Temp (F)"],
            "rows": [
                ["Phoenix", "118"],
                ["Las Vegas", "115"],
                ["Death Valley", "130"],
                ["Tucson", "112"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the column to maximize: High Temp (F).\n"
            "Step 2: Collect all temperature values: 118, 115, 130, 112.\n"
            "Step 3: Find the maximum: 130 (Death Valley).\n"
            "Conclusion: The highest temperature recorded is 130."
        ),
        "answer": "130",
    },
    # 6. Comparison
    {
        "question": "Does team Lions have more wins than team Tigers?",
        "table": {
            "header": ["Team", "Wins", "Losses"],
            "rows": [
                ["Lions", "10", "4"],
                ["Tigers", "7", "7"],
                ["Bears", "10", "4"],
            ],
        },
        "reasoning": (
            "Step 1: Look up Wins for Lions: 10.\n"
            "Step 2: Look up Wins for Tigers: 7.\n"
            "Step 3: Compare: 10 > 7, so Lions have more wins.\n"
            "Conclusion: Yes, Lions have more wins than Tigers."
        ),
        "answer": "yes",
    },
    # 7. Superlative: min
    {
        "question": "Which country has the smallest area?",
        "table": {
            "header": ["Country", "Area (sq km)"],
            "rows": [
                ["Brazil", "8515767"],
                ["Canada", "9984670"],
                ["Australia", "7692024"],
                ["India", "3287263"],
            ],
        },
        "reasoning": (
            "Step 1: Identify the column to minimize: Area (sq km).\n"
            "Step 2: Collect all area values: 8515767, 9984670, 7692024, 3287263.\n"
            "Step 3: Find the minimum: 3287263 (India).\n"
            "Conclusion: India has the smallest area."
        ),
        "answer": "India",
    },
    # 8. Multi-hop: cheapest product's rating
    {
        "question": "What is the rating of the cheapest product?",
        "table": {
            "header": ["Product", "Price", "Rating"],
            "rows": [
                ["Alpha", "50", "4.2"],
                ["Beta", "30", "3.8"],
                ["Gamma", "45", "4.5"],
            ],
        },
        "reasoning": (
            "Step 1: Find the cheapest product by minimizing the Price column.\n"
            "  - Alpha: 50, Beta: 30, Gamma: 45. Minimum price = 30 (Beta).\n"
            "Step 2: Look up the Rating for Beta: 3.8.\n"
            "Conclusion: The rating of the cheapest product (Beta) is 3.8."
        ),
        "answer": "3.8",
    },
]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def get_exemplars(style: str) -> list[dict]:
    """Return the exemplar list for the given style.

    Parameters
    ----------
    style : {"plain", "structured"}

    Returns
    -------
    list[dict]
        The matching exemplar list.
    """
    if style == "plain":
        return EXEMPLARS_PLAIN
    if style == "structured":
        return EXEMPLARS_STRUCTURED
    raise ValueError(f"Unknown style {style!r}. Choose 'plain' or 'structured'.")


def format_exemplar(ex: dict, style: str) -> str:  # noqa: ARG001  (style is informational)
    """Format a single exemplar as a demonstration string.

    The *style* parameter is informational only; the exemplar already carries
    the correct reasoning text for its list.

    The returned string always ends with the line ``Answer: <answer>``.

    Parameters
    ----------
    ex : dict
        A single exemplar dict with keys question, table, reasoning, answer.
    style : str
        One of "plain" or "structured" (informational, not used for rendering).

    Returns
    -------
    str
        A formatted demonstration string.
    """
    serialized = serialize_table(ex["table"])
    lines = [
        f"Question: {ex['question']}",
        "Table:",
        serialized,
        f"Reasoning: {ex['reasoning']}",
        f"Answer: {ex['answer']}",
    ]
    return "\n".join(lines)


In [ ]:
%%writefile src/trace_templates.py
"""Rule-based reasoning-trace generator for Condition C (trace fine-tuning).

Only returns a trace when the derivation is genuinely unambiguous — a wrong
trace used as a training target poisons the experiment, so conservative is
correct.  When in doubt, return None.

Public API
----------
generate_trace(example: dict) -> str | None
    Returns a short reasoning-chain string ending in "Answer: <gold>", or None.

trace_coverage(examples: list[dict]) -> dict
    Returns {"total": N, "with_trace": k, "coverage": k/N}.
"""

from __future__ import annotations


# --------------------------------------------------------------------------- #
# Internal helpers
# --------------------------------------------------------------------------- #

def _gold(example: dict) -> str:
    """Return the canonical gold answer string for an example."""
    answers = example.get("answers")
    if answers and len(answers) > 0:
        return answers[0]
    return example["answer"]


def _normalize(text: str) -> str:
    """Lowercase and strip for cell-matching purposes."""
    return text.lower().strip()


def _find_cell_matches(table: dict, norm_gold: str) -> list[tuple[int, int]]:
    """Return list of (row_index, col_index) where the normalized cell == norm_gold.

    Row index is 0-based into table["rows"]; col_index into table["header"].
    """
    matches: list[tuple[int, int]] = []
    for row_idx, row in enumerate(table["rows"]):
        for col_idx, cell in enumerate(row):
            if _normalize(str(cell)) == norm_gold:
                matches.append((row_idx, col_idx))
    return matches


# --------------------------------------------------------------------------- #
# Rule 1: Unique single-cell lookup
# --------------------------------------------------------------------------- #

def _rule_unique_lookup(example: dict) -> str | None:
    """Emit a trace when the gold value appears in exactly one cell."""
    gold = _gold(example)
    norm_gold = _normalize(gold)
    if not norm_gold:
        return None

    table = example["table"]
    matches = _find_cell_matches(table, norm_gold)

    if len(matches) != 1:
        # Zero matches (e.g. computed answer not literally in table) or >1 — ambiguous.
        return None

    _row_idx, col_idx = matches[0]
    header = table["header"]
    col_name = str(header[col_idx])

    return (
        f"Scan the table for '{gold}'. "
        f"It appears once, in column '{col_name}'. "
        f"Answer: {gold}"
    )


# --------------------------------------------------------------------------- #
# Rule 2: Count-all rows
# --------------------------------------------------------------------------- #

def _rule_count_all_rows(example: dict) -> str | None:
    """Emit a trace when the question asks 'how many' and gold == total row count."""
    question = example.get("question", "")
    if not question.lower().startswith("how many"):
        return None

    gold = _gold(example)
    norm_gold = gold.strip()

    # Gold must be an integer string.
    try:
        gold_int = int(norm_gold)
    except ValueError:
        return None

    n_rows = len(example["table"]["rows"])
    if gold_int != n_rows:
        # Only emit when the answer unambiguously equals the full row count.
        return None

    return (
        f"The question asks how many. "
        f"Count the rows in the table: there are {n_rows}. "
        f"Answer: {norm_gold}"
    )


# --------------------------------------------------------------------------- #
# Public API
# --------------------------------------------------------------------------- #

def generate_trace(example: dict) -> str | None:
    """Return a verifiable reasoning trace, or None if no rule applies.

    Rules are tried in priority order; the first that fires wins.

    Parameters
    ----------
    example : dict
        A unified WTQ example dict (see src/data.py for schema).

    Returns
    -------
    str
        A short reasoning chain ending in "Answer: <gold>".
    None
        When no rule yields an unambiguous derivation.
    """
    # Rule 1: unique single-cell lookup
    trace = _rule_unique_lookup(example)
    if trace is not None:
        return trace

    # Rule 2: count-all rows
    trace = _rule_count_all_rows(example)
    if trace is not None:
        return trace

    # No rule fired.
    return None


def trace_coverage(examples: list[dict]) -> dict:
    """Report what fraction of examples received a trace.

    Parameters
    ----------
    examples : list[dict]
        A list of unified example dicts.

    Returns
    -------
    dict with keys "total", "with_trace", "coverage".
    """
    total = len(examples)
    with_trace = sum(1 for ex in examples if generate_trace(ex) is not None)
    coverage = with_trace / total if total > 0 else 0.0
    return {"total": total, "with_trace": with_trace, "coverage": coverage}


In [ ]:
%%writefile src/metrics.py
"""Evaluation metrics for ECS 111 Final Project.

Pure Python (stdlib + scipy + scikit-learn only). Safe to import without
torch/transformers/datasets.
"""

from __future__ import annotations

import re
import string
from collections import Counter

from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score


# --------------------------------------------------------------------------- #
# Text normalization
# --------------------------------------------------------------------------- #

def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace, strip.

    Faithful to proposal: lowercasing and punctuation removal.
    Articles are intentionally kept (not removed).
    """
    s = s.lower()
    # Remove all punctuation characters
    s = s.translate(str.maketrans("", "", string.punctuation))
    # Collapse any run of whitespace to a single space and strip ends
    s = " ".join(s.split())
    return s


# --------------------------------------------------------------------------- #
# Exact Match
# --------------------------------------------------------------------------- #

def exact_match(pred: str, gold: str) -> bool:
    """Return True iff the normalized prediction equals the normalized gold."""
    return normalize_answer(pred) == normalize_answer(gold)


def exact_match_score(preds: list[str], golds: list[str]) -> float:
    """Mean exact_match over two equal-length lists (returns 0.0–1.0).

    Raises ValueError if the lists have different lengths.
    """
    if len(preds) != len(golds):
        raise ValueError(
            f"preds and golds must have the same length, "
            f"got {len(preds)} vs {len(golds)}"
        )
    if not preds:
        return 0.0
    return sum(exact_match(p, g) for p, g in zip(preds, golds)) / len(preds)


# --------------------------------------------------------------------------- #
# Token F1 (SQuAD-style)
# --------------------------------------------------------------------------- #

def token_f1(pred: str, gold: str) -> float:
    """SQuAD-style token-level F1 over normalized whitespace tokens.

    If either side has zero tokens:
      - both empty -> 1.0
      - one empty  -> 0.0

    common = size of the multiset intersection;
    precision = common / len(pred_tokens);
    recall    = common / len(gold_tokens);
    F1        = 2 * P * R / (P + R), or 0.0 when common == 0.
    """
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)
    common = sum((pred_counter & gold_counter).values())

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1


def token_f1_score(preds: list[str], golds: list[str]) -> float:
    """Mean token_f1 over two equal-length lists (returns 0.0–1.0).

    Raises ValueError if the lists have different lengths.
    """
    if len(preds) != len(golds):
        raise ValueError(
            f"preds and golds must have the same length, "
            f"got {len(preds)} vs {len(golds)}"
        )
    if not preds:
        return 0.0
    return sum(token_f1(p, g) for p, g in zip(preds, golds)) / len(preds)


# --------------------------------------------------------------------------- #
# TabFact label mapping
# --------------------------------------------------------------------------- #

_TRUE_KEYWORDS = ("true", "yes", "entail", "supported", "correct")
_FALSE_KEYWORDS = ("false", "no", "refut", "contradict", "incorrect")

# Pre-compiled patterns anchored at word boundaries (start of keyword only).
# Using \b at the start prevents partial suffix matches (e.g. "correct" inside
# "incorrect"), while omitting \b at the end allows prefix/stem matches
# (e.g. "entail" matches "entailed", "refut" matches "refuted").
_TRUE_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(kw) for kw in _TRUE_KEYWORDS) + r")"
)
_FALSE_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(kw) for kw in _FALSE_KEYWORDS) + r")"
)


def map_tabfact_label(text: str) -> str | None:
    """Map free-form model output to "true", "false", or None.

    Searches the lowercased text for keyword signals using word boundaries:
      "true" side : true / yes / entail / supported / correct
      "false" side: false / no / refut / contradict / incorrect

    If both sides match (ambiguous) or neither matches, returns None.
    Word-boundary matching prevents "correct" from matching inside "incorrect",
    or "no" from matching inside "know".
    """
    lowered = text.lower()
    has_true = bool(_TRUE_PATTERN.search(lowered))
    has_false = bool(_FALSE_PATTERN.search(lowered))

    if has_true and not has_false:
        return "true"
    if has_false and not has_true:
        return "false"
    # ambiguous (both) or neither -> unmappable
    return None


# --------------------------------------------------------------------------- #
# Classification accuracy (TabFact)
# --------------------------------------------------------------------------- #

def classification_accuracy(preds: list[str], golds: list[str]) -> float:
    """Accuracy for TabFact binary classification.

    golds must already be "true" or "false".
    Each pred is mapped via map_tabfact_label; unmappable preds count as wrong.

    Raises ValueError if the lists have different lengths.
    """
    if len(preds) != len(golds):
        raise ValueError(
            f"preds and golds must have the same length, "
            f"got {len(preds)} vs {len(golds)}"
        )
    if not preds:
        return 0.0
    correct = sum(
        map_tabfact_label(p) == g for p, g in zip(preds, golds)
    )
    return correct / len(preds)


# --------------------------------------------------------------------------- #
# McNemar test (paired system comparison)
# --------------------------------------------------------------------------- #

def mcnemar_test(
    correct_a: list[bool], correct_b: list[bool]
) -> dict:
    """Paired McNemar significance test between two systems.

    b = count(a correct, b wrong)
    c = count(a wrong,   b correct)

    Uses an exact binomial test on the discordant pairs:
      scipy.stats.binomtest(min(b, c), b + c, 0.5)

    If b + c == 0 (no discordant pairs), pvalue is 1.0.

    Returns {"b": b, "c": c, "n_discordant": b+c, "pvalue": float}.

    Raises ValueError if the lists have different lengths.
    """
    if len(correct_a) != len(correct_b):
        raise ValueError(
            f"correct_a and correct_b must have the same length, "
            f"got {len(correct_a)} vs {len(correct_b)}"
        )

    b = sum(1 for a, bb in zip(correct_a, correct_b) if a and not bb)
    c = sum(1 for a, bb in zip(correct_a, correct_b) if not a and bb)
    n_discordant = b + c

    if n_discordant == 0:
        pvalue = 1.0
    else:
        result = binomtest(min(b, c), n_discordant, 0.5)
        pvalue = float(result.pvalue)

    return {"b": b, "c": c, "n_discordant": n_discordant, "pvalue": pvalue}


# --------------------------------------------------------------------------- #
# Cohen's kappa
# --------------------------------------------------------------------------- #

def cohen_kappa(rater_a: list[int], rater_b: list[int]) -> float:
    """Inter-rater agreement using sklearn.metrics.cohen_kappa_score.

    Ratings are expected to be 0, 1, or 2.
    Returns kappa in [-1, 1]; perfect agreement -> 1.0.
    """
    return float(cohen_kappa_score(rater_a, rater_b))


In [ ]:
%%writefile src/trainer.py
"""Seq2seq fine-tuning + generation for Flan-T5.

A small, explicit training loop (rather than the framework Trainer) so behaviour
is identical across transformers versions and on cuda / mps / cpu. Used by
Conditions B (answers-only) and C (reasoning traces). `train` takes already-built
source/target strings, keeping this module independent of prompt/data specifics.
"""

from __future__ import annotations

import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from . import config


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class _Seq2SeqDataset(Dataset):
    def __init__(self, sources: list[str], targets: list[str]):
        assert len(sources) == len(targets)
        self.sources = sources
        self.targets = targets

    def __len__(self) -> int:
        return len(self.sources)

    def __getitem__(self, i: int) -> dict:
        return {"source": self.sources[i], "target": self.targets[i]}


def _make_collate(tokenizer):
    def collate(batch: list[dict]) -> dict:
        sources = [b["source"] for b in batch]
        targets = [b["target"] for b in batch]
        model_inputs = tokenizer(
            sources,
            max_length=config.MAX_SOURCE_LEN,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        labels = tokenizer(
            text_target=targets,
            max_length=config.MAX_TARGET_LEN,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )["input_ids"]
        # Ignore pad tokens in the loss.
        labels[labels == tokenizer.pad_token_id] = -100
        model_inputs["labels"] = labels
        return model_inputs

    return collate


def load_model_and_tokenizer(model_id: str, device: str | None = None):
    device = device or config.get_device()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
    return model, tokenizer, device


def train(
    model_id: str,
    sources: list[str],
    targets: list[str],
    seed: int,
    smoke: bool = False,
    device: str | None = None,
):
    """Fine-tune `model_id` on (sources -> targets). Returns (model, tokenizer, device).

    smoke=True shrinks to flan-t5-small + a couple of steps for a real but fast
    end-to-end check (used locally on MPS and as the first Colab sanity cell).
    """
    set_seed(seed)
    if smoke:
        model_id = config.SMOKE_MODEL
        sources = sources[: config.SMOKE_TRAIN_N]
        targets = targets[: config.SMOKE_TRAIN_N]

    model, tokenizer, device = load_model_and_tokenizer(model_id, device)
    loader = DataLoader(
        _Seq2SeqDataset(sources, targets),
        batch_size=config.TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=_make_collate(tokenizer),
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE)

    epochs = config.SMOKE_EPOCHS if smoke else config.EPOCHS
    max_steps = config.SMOKE_MAX_STEPS if smoke else None

    model.train()
    step = 0
    optimizer.zero_grad()
    for _epoch in range(epochs):
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss / config.GRAD_ACCUM_STEPS
            loss.backward()
            if (i + 1) % config.GRAD_ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                step += 1
                if max_steps is not None and step >= max_steps:
                    break
        if max_steps is not None and step >= max_steps:
            break
    # Flush any remaining accumulated gradients.
    optimizer.step()
    optimizer.zero_grad()
    return model, tokenizer, device


@torch.no_grad()
def generate(
    model,
    tokenizer,
    sources: list[str],
    device: str | None = None,
    batch_size: int = 16,
    max_source_len: int | None = None,
    truncation_side: str = "right",
) -> tuple[list[str], float]:
    """Greedy-decode predictions for `sources`. Returns (predictions, seconds_per_example).

    truncation_side='left' keeps the END of the prompt. CoT prompts put the real
    question after the exemplars, so they must truncate from the left to avoid
    cutting the question off.
    """
    device = device or config.get_device()
    max_source_len = max_source_len or config.MAX_SOURCE_LEN
    saved_side = tokenizer.truncation_side
    tokenizer.truncation_side = truncation_side
    model.eval()
    preds: list[str] = []
    start = time.perf_counter()
    for i in range(0, len(sources), batch_size):
        chunk = sources[i : i + batch_size]
        inputs = tokenizer(
            chunk,
            max_length=max_source_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)
        out = model.generate(
            **inputs,
            max_new_tokens=config.GEN_MAX_NEW_TOKENS,
            do_sample=False,  # greedy, deterministic
            num_beams=1,
        )
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    elapsed = time.perf_counter() - start
    per_example = elapsed / max(len(sources), 1)
    tokenizer.truncation_side = saved_side  # restore
    return preds, per_example


def peak_memory_mb(device: str) -> float | None:
    """Peak allocated memory in MB for the run, if the backend reports it."""
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / 1024**2
    if device == "mps" and hasattr(torch.mps, "current_allocated_memory"):
        return torch.mps.current_allocated_memory() / 1024**2
    return None


In [ ]:
%%writefile src/evaluate.py
"""Run a condition end-to-end: predict -> metrics -> error labels -> results JSON.

Produces one JSON per (condition, model, seed) under results/. analysis.py
aggregates those files across seeds and runs the statistical tests.
"""

from __future__ import annotations

import json
from pathlib import Path

from . import config
from .metrics import (
    exact_match,
    exact_match_score,
    map_tabfact_label,
    token_f1_score,
)
from .prompts import extract_answer
from .trainer import generate, peak_memory_mb

# Keyword cues for the WTQ error-type heuristic (proposal section 6).
_AGG_CUES = (
    "how many", "how much", "number of", "count", "total", "sum", "average",
    "mean", "most", "least", "largest", "smallest", "highest", "lowest",
    "maximum", "minimum", "max", "min", "longest", "shortest",
)
_MULTIHOP_CUES = (" and ", " after ", " before ", " than ", " also ", " both ", " then ")


def classify_question(question: str) -> str:
    """Heuristic reasoning type for an INCORRECT WTQ prediction.

    Returns one of "aggregation" | "multi_hop" | "lookup". This is an
    approximate label (proposal calls for a 3-way error breakdown); it keys off
    surface cues in the question, not gold logic.
    """
    q = f" {question.lower()} "
    if any(cue in q for cue in _AGG_CUES):
        return "aggregation"
    if any(cue in q for cue in _MULTIHOP_CUES):
        return "multi_hop"
    return "lookup"


def label_error(example: dict, correct: bool) -> str:
    """Per-example label: 'correct' or one of the three WTQ error types."""
    if correct:
        return "correct"
    return classify_question(example["question"])


def _is_correct(pred: str, example: dict, task: str) -> bool:
    if task == "tabfact":
        return map_tabfact_label(pred) == example["answer"]
    return exact_match(pred, example["answer"])


def save_results(results: dict) -> Path:
    config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    model_short = results["model"].split("/")[-1]
    path = config.RESULTS_DIR / f"{results['condition']}_{model_short}_seed{results['seed']}.json"
    path.write_text(json.dumps(results, indent=2))
    return path


def predict_and_evaluate(
    model,
    tokenizer,
    examples: list[dict],
    prompt_fn,
    *,
    condition: str,
    model_id: str,
    seed: int,
    task: str,
    device: str | None = None,
    batch_size: int = 16,
    save: bool = True,
) -> dict:
    """Generate predictions for `examples`, score them, label errors, and assemble results.

    prompt_fn: example -> source string (baseline or CoT builder).
    task: "wtq" (EM + token-F1) or "tabfact" (classification accuracy).
    """
    device = device or config.get_device()
    is_cot = condition.startswith("cot")
    # CoT prompts are long and put the question last -> truncate from the left.
    max_source_len = config.MAX_SOURCE_LEN_PROMPT if is_cot else config.MAX_SOURCE_LEN
    truncation_side = "left" if is_cot else "right"

    sources = [prompt_fn(ex) for ex in examples]
    raw_preds, seconds_per_example = generate(
        model, tokenizer, sources, device, batch_size,
        max_source_len=max_source_len, truncation_side=truncation_side,
    )
    preds = [extract_answer(p) for p in raw_preds]
    golds = [ex["answer"] for ex in examples]

    records = []
    for ex, pred, raw in zip(examples, preds, raw_preds):
        correct = _is_correct(pred, ex, task)
        rec = {
            "id": ex["id"],
            "pred": pred,
            "gold": ex["answer"],
            "correct": correct,
            "error_type": label_error(ex, correct),
        }
        if is_cot:
            rec["raw"] = raw  # full generation incl. reasoning, for chain-quality rating
        records.append(rec)

    if task == "tabfact":
        n_correct = sum(r["correct"] for r in records)
        metrics = {"classification_accuracy": n_correct / max(len(records), 1)}
        error_distribution = None
    else:
        metrics = {
            "exact_match": exact_match_score(preds, golds),
            "token_f1": token_f1_score(preds, golds),
        }
        error_distribution = {et: 0 for et in config.ERROR_TYPES}
        for r in records:
            error_distribution[r["error_type"]] += 1

    results = {
        "condition": condition,
        "model": model_id,
        "seed": seed,
        "task": task,
        "n": len(examples),
        "metrics": metrics,
        "error_distribution": error_distribution,
        "compute": {
            "seconds_per_example": seconds_per_example,
            "peak_memory_mb": peak_memory_mb(device),
            "device": device,
        },
        "predictions": records,
    }
    if save:
        save_results(results)
    return results


In [ ]:
%%writefile src/analysis.py
"""Aggregate per-(condition, seed) result JSONs into the reported numbers.

- mean +/- std of the primary metric across seeds, per condition
- McNemar's test between two conditions (paired per-example correctness)
- Cohen's kappa between two chain-quality raters
- summary table (CSV + Markdown) and bar plots

Pure aggregation over the JSON schema written by evaluate.py; no torch needed.
"""

from __future__ import annotations

import json
import statistics
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless: save figures, never open a window
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from . import config  # noqa: E402
from .metrics import cohen_kappa, mcnemar_test  # noqa: E402


def primary_metric(result: dict) -> tuple[str, float]:
    """The headline metric for a result: EM (wtq) or classification accuracy (tabfact)."""
    if result["task"] == "tabfact":
        return "classification_accuracy", result["metrics"]["classification_accuracy"]
    return "exact_match", result["metrics"]["exact_match"]


def load_results(results_dir: Path | str = config.RESULTS_DIR) -> list[dict]:
    """Load every per-condition result JSON in results_dir.

    Skips JSON files that are not per-condition results (e.g. chain_quality.json)
    so helper files can sit next to the results without breaking aggregation.
    """
    results_dir = Path(results_dir)
    out = []
    for path in sorted(results_dir.glob("*.json")):
        data = json.loads(path.read_text())
        if isinstance(data, dict) and "condition" in data and "predictions" in data:
            out.append(data)
    return out


def aggregate(results: list[dict]) -> pd.DataFrame:
    """Group results by (condition, model, task); mean/std of primary metric over seeds."""
    groups: dict[tuple, list[dict]] = {}
    for r in results:
        groups.setdefault((r["condition"], r["model"], r["task"]), []).append(r)

    rows = []
    for (condition, model, task), items in groups.items():
        metric_name, _ = primary_metric(items[0])
        values = [primary_metric(r)[1] for r in items]
        spe = [r["compute"]["seconds_per_example"] for r in items]
        rows.append(
            {
                "condition": condition,
                "model": model.split("/")[-1],
                "task": task,
                "metric": metric_name,
                "mean": statistics.fmean(values),
                "std": statistics.pstdev(values) if len(values) > 1 else 0.0,
                "n_seeds": len(values),
                "sec_per_example": statistics.fmean(spe),
            }
        )
    df = pd.DataFrame(rows).sort_values(["task", "condition", "model"]).reset_index(drop=True)
    return df


def _correct_by_id(result: dict) -> dict[str, bool]:
    return {rec["id"]: rec["correct"] for rec in result["predictions"]}


def mcnemar_between(result_a: dict, result_b: dict) -> dict:
    """Paired McNemar on per-example correctness over the ids common to both results."""
    a, b = _correct_by_id(result_a), _correct_by_id(result_b)
    ids = [i for i in a if i in b]
    if not ids:
        raise ValueError("No overlapping example ids between the two results.")
    return mcnemar_test([a[i] for i in ids], [b[i] for i in ids])


def kappa_between_raters(ratings_a: list[int], ratings_b: list[int]) -> float:
    """Cohen's kappa for two chain-quality raters (0/1/2 scores)."""
    return cohen_kappa(ratings_a, ratings_b)


def load_rater_csv(path: Path | str) -> list[int]:
    """Load a rater file: CSV with an 'id' and a 'rating' column, sorted by id."""
    df = pd.read_csv(path).sort_values("id")
    return df["rating"].astype(int).tolist()


def write_summary_table(df: pd.DataFrame, out_dir: Path | str = config.RESULTS_DIR) -> dict[str, Path]:
    """Write the summary table as CSV and Markdown. Returns the two paths."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "summary_table.csv"
    md_path = out_dir / "summary_table.md"
    df.to_csv(csv_path, index=False)
    md_path.write_text(df.to_markdown(index=False))
    return {"csv": csv_path, "md": md_path}


def plot_primary_metric(df: pd.DataFrame, out_path: Path | str) -> Path:
    """Bar chart of mean primary metric per condition (error bars = std across seeds)."""
    out_path = Path(out_path)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    labels = [f"{r.condition}\n({r.model})" for r in df.itertuples()]
    ax.bar(labels, df["mean"], yerr=df["std"], capsize=4)
    ax.set_ylabel("primary metric (EM / classification acc)")
    ax.set_title("Primary metric by condition (mean ± std over seeds)")
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path


def plot_error_distribution(results: list[dict], out_path: Path | str) -> Path:
    """Stacked bar of WTQ error-type counts per condition."""
    out_path = Path(out_path)
    wtq = [r for r in results if r["task"] == "wtq" and r["error_distribution"]]
    types = [t for t in config.ERROR_TYPES if t != "correct"]
    labels = [f"{r['condition']}/{r['model'].split('/')[-1]}/s{r['seed']}" for r in wtq]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bottom = [0] * len(wtq)
    for et in types:
        vals = [r["error_distribution"].get(et, 0) for r in wtq]
        ax.bar(labels, vals, bottom=bottom, label=et)
        bottom = [b + v for b, v in zip(bottom, vals)]
    ax.set_ylabel("error count")
    ax.set_title("WTQ error-type distribution by condition")
    ax.legend()
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path


In [ ]:
%%writefile src/report_fill.py
"""Map every report/slide placeholder token to a real value from results/*.json.

Single source of truth so the report and the slides can never disagree, and so
every printed number traces back to a results JSON file (no hand-typed numbers).
Pure dict math over the schema written by evaluate.py; safe to import without torch.
"""
from __future__ import annotations

from .analysis import mcnemar_between


def _pick(results, condition, model_short=None, task="wtq"):
    return [r for r in results
            if r["condition"] == condition and r["task"] == task
            and (model_short is None or r["model"].split("/")[-1] == model_short)]


def _mean_std(values):
    m = sum(values) / len(values)
    if len(values) < 2:
        return m, 0.0
    var = sum((v - m) ** 2 for v in values) / len(values)  # population std (matches analysis.aggregate)
    return m, var ** 0.5


def _fmt_metric(results, condition, model_short, key):
    rs = _pick(results, condition, model_short)
    if not rs:
        return "n/a"
    m, s = _mean_std([r["metrics"][key] for r in rs])
    return f"{m:.3f} ± {s:.3f}"


def _em_mean(results, condition, model_short):
    rs = _pick(results, condition, model_short)
    return None if not rs else sum(r["metrics"]["exact_match"] for r in rs) / len(rs)


def _tabfact(results, gen_condition):
    rs = _pick(results, gen_condition, "flan-t5-base", task="tabfact")
    if not rs:
        return "n/a"
    m, s = _mean_std([r["metrics"]["classification_accuracy"] for r in rs])
    return f"{m:.3f} ± {s:.3f}"


def _best_base_condition(results):
    cands = ["cot_plain", "cot_structured", "finetune_answers", "finetune_traces"]
    scored = [(c, _em_mean(results, c, "flan-t5-base")) for c in cands]
    scored = [(c, v) for c, v in scored if v is not None]
    return max(scored, key=lambda cv: cv[1])[0] if scored else None


def build_token_map(results, chain_quality):
    """Return {exact_token_string: formatted_value} for report + slides."""
    tm = {}
    # Prompting conditions run on both models; fine-tuning is base only (large OOMs on a T4).
    for cond in ["baseline", "cot_plain", "cot_structured"]:
        for short, model in [("base", "flan-t5-base"), ("large", "flan-t5-large")]:
            tm[f"[EM: {cond} {short}]"] = _fmt_metric(results, cond, model, "exact_match")
            tm[f"[F1: {cond} {short}]"] = _fmt_metric(results, cond, model, "token_f1")
    for cond in ["finetune_answers", "finetune_traces"]:
        tm[f"[EM: {cond} base]"] = _fmt_metric(results, cond, "flan-t5-base", "exact_match")
        tm[f"[F1: {cond} base]"] = _fmt_metric(results, cond, "flan-t5-base", "token_f1")

    # TabFact generalization (only the two fine-tuned models are evaluated on TabFact).
    for cond in ["finetune_answers", "finetune_traces"]:
        acc = _tabfact(results, f"generalization_{cond}")
        tm[f"[ACC: tabfact {cond} base]"] = acc
        tm[f"[TabFact acc: {cond}]"] = acc
    # Optional baseline floor on TabFact, only if it was actually run.
    tm["[ACC: tabfact baseline base]"] = _tabfact(results, "generalization_baseline")

    # Best TabFact across the conditions we have.
    flat = []
    for cond in ("finetune_answers", "finetune_traces"):
        flat += [r["metrics"]["classification_accuracy"]
                 for r in _pick(results, f"generalization_{cond}", "flan-t5-base", "tabfact")]
    tm["[ACC: tabfact best]"] = f"{max(flat):.3f}" if flat else "n/a"

    # Success criterion 1: best base condition EM minus baseline base EM.
    base_em = _em_mean(results, "baseline", "flan-t5-base")
    best_c = _best_base_condition(results)
    if base_em is not None and best_c is not None:
        gap = (_em_mean(results, best_c, "flan-t5-base") - base_em) * 100
        tm["[EM_GAP: best vs baseline]"] = f"{gap:+.1f} EM points ({best_c})"
    else:
        tm["[EM_GAP: best vs baseline]"] = "n/a"

    # Error-type shares for the headline (best base) condition, summed over seeds.
    err = {"lookup": 0, "aggregation": 0, "multi_hop": 0}
    for r in _pick(results, best_c or "baseline", "flan-t5-base"):
        for k in err:
            err[k] += (r["error_distribution"] or {}).get(k, 0)
    tot = sum(err.values()) or 1
    tm["[ERR: lookup]"] = f"{100 * err['lookup'] / tot:.0f}%"
    tm["[ERR: aggregation]"] = f"{100 * err['aggregation'] / tot:.0f}%"
    tm["[ERR: multihop]"] = f"{100 * err['multi_hop'] / tot:.0f}%"

    # McNemar: best CoT base vs best fine-tune base, seed 13 (paired per-example).
    cot_best = max(["cot_plain", "cot_structured"],
                   key=lambda c: _em_mean(results, c, "flan-t5-base") or -1)
    ft_best = max(["finetune_answers", "finetune_traces"],
                  key=lambda c: _em_mean(results, c, "flan-t5-base") or -1)
    a = _pick(results, cot_best, "flan-t5-base")
    b = _pick(results, ft_best, "flan-t5-base")
    a13 = next((r for r in a if r["seed"] == 13), a[0] if a else None)
    b13 = next((r for r in b if r["seed"] == 13), b[0] if b else None)
    if a13 and b13:
        p = mcnemar_between(a13, b13)["pvalue"]
        tm["[MCNEMAR: cot vs finetune]"] = f"p = {p:.3f} ({cot_best} vs {ft_best}, seed 13)"
    else:
        tm["[MCNEMAR: cot vs finetune]"] = "n/a"

    # Chain quality (from scripts/rate_chains.py output).
    tm["[KAPPA: chain quality]"] = f"{chain_quality['kappa']:.3f}"

    # Per-condition wall-clock ceiling (max sec/example * n over conditions), minutes.
    per = [r["compute"]["seconds_per_example"] * r["n"] for r in results]
    tm["[TIME: per condition]"] = f"~{max(per) / 60:.0f} min" if per else "n/a"
    return tm


In [ ]:
%%writefile scripts/run_all_local.py
"""Run every condition for real on this machine and save results.

This produces REAL numbers (not smoke): baseline, CoT (plain + structured),
fine-tune answers-only, fine-tune with traces, and the TabFact generalization
test. It then aggregates everything into a summary table and plots.

Two scales:
  --scale quick   one seed, small slices. ~30-45 min. For a first real look.
  --scale full    config scale (2 seeds, 1000 eval, 8000 train). Hours. Final.

Usage:
    python scripts/run_all_local.py --scale quick
Results land in results/ . Re-running overwrites per (condition, model, seed).
"""

from __future__ import annotations

import argparse
import functools
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src import config
from src.data import load_tabfact, load_wtq_eval, load_wtq_train
from src.evaluate import predict_and_evaluate
from src.prompts import (
    build_baseline_prompt,
    build_cot_prompt,
    build_tabfact_prompt,
    build_train_source,
    build_train_target,
)
from src.trace_templates import generate_trace
from src.trainer import load_model_and_tokenizer, train


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def results_exist(condition: str, model_id: str, seed: int) -> bool:
    """True if this (condition, model, seed) already has a saved result (resume support)."""
    model_short = model_id.split("/")[-1]
    return (config.RESULTS_DIR / f"{condition}_{model_short}_seed{seed}.json").exists()


def get_settings(scale: str) -> dict:
    if scale == "full":
        return {
            "prompt_models": config.PROMPT_MODELS,
            "seeds": config.SEEDS,
            "eval_n": config.EVAL_N,
            "eval_n_cot": config.EVAL_N_COT,
            "train_n": config.TRAIN_N,
        }
    # quick: one seed, small slices, base + large still both prompted
    return {
        "prompt_models": config.PROMPT_MODELS,
        "seeds": [13],
        "eval_n": 120,
        "eval_n_cot": 80,
        "train_n": 1200,
    }


def run_baseline(s: dict, device: str) -> None:
    log("=== BASELINE ===")
    for model_id in s["prompt_models"]:
        todo = [seed for seed in s["seeds"] if not results_exist("baseline", model_id, seed)]
        if not todo:
            log(f"baseline {model_id.split('/')[-1]}: all seeds done, skip")
            continue
        model, tok, device = load_model_and_tokenizer(model_id, device)
        for seed in todo:
            ex = load_wtq_eval(n=s["eval_n"], seed=seed)
            res = predict_and_evaluate(
                model, tok, ex, build_baseline_prompt,
                condition="baseline", model_id=model_id, seed=seed,
                task="wtq", device=device,
            )
            log(f"baseline {model_id.split('/')[-1]} seed{seed}: {res['metrics']}")


def run_baseline_tabfact_floor(s: dict, device: str) -> None:
    """Cheap OOD floor: the un-fine-tuned base on TabFact, for reference on the slides."""
    log("=== BASELINE TabFact FLOOR ===")
    todo = [seed for seed in s["seeds"]
            if not results_exist("generalization_baseline", config.FINETUNE_MODEL, seed)]
    if not todo:
        log("generalization_baseline: all seeds done, skip")
        return
    tf = load_tabfact(n=s["eval_n"], seed=config.EVAL_SEED)
    model, tok, device = load_model_and_tokenizer(config.FINETUNE_MODEL, device)
    for seed in todo:
        res = predict_and_evaluate(
            model, tok, tf, build_tabfact_prompt,
            condition="generalization_baseline", model_id=config.FINETUNE_MODEL, seed=seed,
            task="tabfact", device=device,
        )
        log(f"generalization_baseline seed{seed}: {res['metrics']}")


def run_cot(s: dict, device: str, styles=("plain", "structured")) -> None:
    log("=== CHAIN OF THOUGHT ===")
    for style in styles:
        for model_id in s["prompt_models"]:
            todo = [seed for seed in s["seeds"] if not results_exist(f"cot_{style}", model_id, seed)]
            if not todo:
                log(f"cot_{style} {model_id.split('/')[-1]}: all seeds done, skip")
                continue
            model, tok, device = load_model_and_tokenizer(model_id, device)
            for seed in todo:
                ex = load_wtq_eval(n=s["eval_n_cot"], seed=seed)
                pf = functools.partial(build_cot_prompt, style=style, n_shots=config.N_SHOTS)
                res = predict_and_evaluate(
                    model, tok, ex, pf,
                    condition=f"cot_{style}", model_id=model_id, seed=seed,
                    task="wtq", device=device,
                )
                log(f"cot_{style} {model_id.split('/')[-1]} seed{seed}: {res['metrics']}")


def run_finetune(s: dict, device: str, condition: str, use_traces: bool) -> None:
    log(f"=== FINE TUNE: {condition} ===")
    Path("checkpoints").mkdir(exist_ok=True)
    for seed in s["seeds"]:
        ckpt = f"checkpoints/{condition}_seed{seed}"
        if results_exist(condition, config.FINETUNE_MODEL, seed) and Path(ckpt).exists():
            log(f"{condition} seed{seed}: results + checkpoint exist, skip")
            continue
        train_ex = load_wtq_train(n=s["train_n"], seed=seed)
        sources = [build_train_source(e) for e in train_ex]
        if use_traces:
            targets = [build_train_target(e, trace=generate_trace(e)) for e in train_ex]
            n_tr = sum(generate_trace(e) is not None for e in train_ex)
            log(f"{condition} seed{seed}: {n_tr}/{len(train_ex)} train rows got a reasoning chain")
        else:
            targets = [build_train_target(e) for e in train_ex]
        log(f"{condition} seed{seed}: training on {len(train_ex)} examples ...")
        model, tok, device = train(config.FINETUNE_MODEL, sources, targets, seed=seed, device=device)
        ckpt = f"checkpoints/{condition}_seed{seed}"
        model.save_pretrained(ckpt); tok.save_pretrained(ckpt)
        eval_ex = load_wtq_eval(n=s["eval_n"], seed=config.EVAL_SEED)
        res = predict_and_evaluate(
            model, tok, eval_ex, build_baseline_prompt,
            condition=condition, model_id=config.FINETUNE_MODEL, seed=seed,
            task="wtq", device=device,
        )
        log(f"{condition} seed{seed}: {res['metrics']}")


def run_generalization(s: dict, device: str, only=None) -> None:
    log("=== GENERALIZATION (TabFact) ===")
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    conds = only if only is not None else ["finetune_answers", "finetune_traces"]
    tf = load_tabfact(n=s["eval_n"], seed=config.EVAL_SEED)
    for cond in conds:
        for seed in s["seeds"]:
            gen_cond = f"generalization_{cond}"
            if results_exist(gen_cond, config.FINETUNE_MODEL, seed):
                log(f"{gen_cond} seed{seed}: results exist, skip")
                continue
            ckpt = f"checkpoints/{cond}_seed{seed}"
            if not Path(ckpt).exists():
                log(f"skip {cond} seed{seed}: no checkpoint")
                continue
            model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device)
            tok = AutoTokenizer.from_pretrained(ckpt)
            res = predict_and_evaluate(
                model, tok, tf, build_tabfact_prompt,
                condition=f"generalization_{cond}", model_id=config.FINETUNE_MODEL, seed=seed,
                task="tabfact", device=device,
            )
            log(f"generalization_{cond} seed{seed}: {res['metrics']}")


def aggregate() -> None:
    log("=== AGGREGATE ===")
    from src import analysis

    results = analysis.load_results()
    df = analysis.aggregate(results)
    paths = analysis.write_summary_table(df)
    analysis.plot_primary_metric(df, config.RESULTS_DIR / "primary_metric.png")
    if any(r["task"] == "wtq" for r in results):
        analysis.plot_error_distribution(results, config.RESULTS_DIR / "error_distribution.png")
    log(f"wrote summary table -> {paths['md']}")
    print("\n================ SUMMARY ================\n")
    print(df.to_string(index=False))
    print("\nFiles in results/: summary_table.csv, summary_table.md, primary_metric.png, error_distribution.png")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--scale", choices=["quick", "full"], default="quick")
    ap.add_argument(
        "--shard",
        default="all",
        choices=["all", "baseline", "cot_plain", "cot_structured",
                 "finetune_answers", "finetune_traces"],
        help="Run one slice of the work (lets teammates split the run across Colab accounts).",
    )
    args = ap.parse_args()

    device = config.get_device()
    s = get_settings(args.scale)
    sh = args.shard
    log(f"device={device} scale={args.scale} shard={sh} settings={s}")
    t0 = time.time()

    # Each shard is self-contained: a fine-tune shard also runs its own TabFact
    # generalization, so no shard depends on another shard's checkpoint.
    if sh in ("all", "baseline"):
        run_baseline(s, device)
        run_baseline_tabfact_floor(s, device)
    if sh in ("all", "cot_plain"):
        run_cot(s, device, styles=["plain"])
    if sh in ("all", "cot_structured"):
        run_cot(s, device, styles=["structured"])
    if sh in ("all", "finetune_answers"):
        run_finetune(s, device, "finetune_answers", use_traces=False)
        run_generalization(s, device, only=["finetune_answers"])
    if sh in ("all", "finetune_traces"):
        run_finetune(s, device, "finetune_traces", use_traces=True)
        run_generalization(s, device, only=["finetune_traces"])
    aggregate()

    log(f"ALL DONE in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
%pip install -q torch transformers datasets sentencepiece scipy scikit-learn pandas matplotlib tabulate

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# {SHARD} is expanded by IPython from the Python variable set above.
!python scripts/run_all_local.py --scale full --shard {SHARD}


In [ ]:
import os

path = "results/summary_table.md"
if os.path.exists(path):
    print(open(path).read())
else:
    print("results/summary_table.md not found yet — the run may be partial.")


In [ ]:
import shutil

shutil.make_archive("ecs111_results", "zip", "results")
try:
    from google.colab import files
    files.download("ecs111_results.zip")
except Exception as e:
    print("download unavailable (not on Colab?):", e)
print("Send ecs111_results.zip back, or paste results/summary_table.md.")
